In [2]:
import pandas as pd
from pathlib import Path

In [3]:
def load_data(path):
    columns =["square_id",
          "time_interval",
          "country_code",
          "sms_in",
          "sms_out",
          "call_in",
          "call_out",
          "internet"]
    data = pd.read_csv(path,
                       sep="\t",
                       header=None,
                       names=columns
                      )


    data["time_interval"] = pd.to_datetime(data["time_interval"],unit='ms')
    data["time_interval"] = data["time_interval"].dt.tz_localize("UTC").dt.tz_convert('Europe/Rome')
    data = data.fillna(0)
    data = data.groupby(["square_id","time_interval"]).sum()
    data = data.drop(columns=["country_code"])
    data = data.reset_index()


    return data

In [4]:
def load_total_data(num_file):
  DATA_DIR = Path("data")
  data_list = []
  for i in range(1,num_file+1):


    path = DATA_DIR / f"sms-call-internet-mi-2013-11-{i:02d}.txt"

    try:
      data = load_data(path)
      data_list.append(data)

      print("[",end='')
      print("#"*i,end='')
      print("="*(20-i),end='')

      print("]",end='\r')
    except Exception as e:
      print(f"failed at {i} : {e} ")


  total_data = pd.concat(data_list,axis=0,ignore_index=True)
  return total_data

In [5]:
def cell_position(id):

  row = (id-1) // 100
  column = (id-1) % 100


  return row,column

In [20]:
def position_to_cell(row,column):
    cid = row *100 + column+ 1
    return cid

In [11]:
total_data = load_total_data(21)

[#####################]

In [14]:
total_data.groupby("square_id")["internet"].sum()

square_id
1         31064.400686
2         31201.632549
3         31347.710439
4         30666.906119
5         27979.373911
             ...      
9996     116330.365691
9997     126081.767946
9998     124258.150176
9999      79581.192889
10000     62721.012632
Name: internet, Length: 10000, dtype: float64

In [26]:
center_cell = position_to_cell(*cell_position(5161))

5161

In [85]:
center_cell = cell_position(5161)

radius = 10
center_positions = []
for i in range(center_cell[0]-radius,center_cell[0]+radius):
    for j in range(center_cell[1]-radius,center_cell[1]+radius):
        cid = position_to_cell(i,j)
        center_positions.append(cid)

In [86]:
total_data[total_data['square_id'].isin(center_positions)]

,square_id,time_interval,sms_in,sms_out,call_in,call_out,internet
597600,4151,2013-11-01 00:00:00+01:00,7.947718,3.125655,3.184955,3.753729,120.654932
597601,4151,2013-11-01 00:10:00+01:00,4.062705,1.310449,1.882983,4.261470,119.137933
597602,4151,2013-11-01 00:20:00+01:00,3.252442,1.554408,2.704398,3.302103,132.895668
597603,4151,2013-11-01 00:30:00+01:00,4.274271,1.348062,0.686236,2.313219,111.203897
597604,4151,2013-11-01 00:40:00+01:00,2.764185,1.537330,0.946893,1.109337,197.078066
...,...,...,...,...,...,...,...
29673565,6070,2013-11-21 23:10:00+01:00,36.859743,9.694463,5.594184,8.328710,466.380082
29673566,6070,2013-11-21 23:20:00+01:00,17.012003,27.965263,2.667454,34.051425,563.590204
29673567,6070,2013-11-21 23:30:00+01:00,22.252982,13.635865,2.366904,7.447729,561.078394
29673568,6070,2013-11-21 23:40:00+01:00,16.759331,12.257659,0.930733,4.376098,424.449335


In [87]:
sub = total_data[total_data['square_id'].isin(center_positions)]

In [92]:
toplam = sub.groupby('square_id')['internet'].sum()

In [93]:
std = sub.groupby('square_id')['internet'].std()

In [95]:
toplam.describe()

count    4.000000e+02
mean     1.340053e+06
std      7.507307e+05
min      1.967342e+05
25%      7.611291e+05
50%      1.281966e+06
75%      1.797563e+06
max      4.365863e+06
Name: internet, dtype: float64

In [96]:
std.describe()

count     400.000000
mean      229.966606
std       182.118672
min        25.257556
25%       110.498451
50%       175.398086
75%       289.875221
max      1273.922579
Name: internet, dtype: float64

In [97]:
def select_valid_cells(data, min_total=1000, min_std=10, verbose=True):
    """
    Modellenebilir hücreleri seçer.
    Kriterler: toplam aktivite eşiği + varyans eşiği.
    Dönüş: (gecerli_id_listesi, elenen_id_listesi)
    """
    total = data.groupby("square_id")["internet"].sum()
    std = data.groupby("square_id")["internet"].std()

    mask = (total >= min_total) & (std >= min_std)

    valid = total[mask].index.tolist()
    dropped = total[~mask].index.tolist()

    if verbose:
        print(f"toplam hücre : {len(total)}")
        print(f"geçerli      : {len(valid)}")
        print(f"elenen       : {len(dropped)}")
        if dropped:
            print(f"elenen id'ler: {dropped[:10]}{' ...' if len(dropped) > 10 else ''}")

    return valid, dropped

In [98]:
valid_cells, dropped_cells = select_valid_cells(sub)
sub = sub[sub["square_id"].isin(valid_cells)]

toplam hücre : 400
geçerli      : 400
elenen       : 0


In [102]:
pivot_internet = sub.pivot_table(index="time_interval", columns="square_id", values="internet")
pivot_internet

square_id,4151,4152,4153,4154,4155,4156,4157,4158,4159,4160,...,6061,6062,6063,6064,6065,6066,6067,6068,6069,6070
time_interval,,,,,,,,,,,,,,,,,,,,,
2013-11-01 00:00:00+01:00,120.654932,138.451424,215.463804,435.474639,600.853735,598.512774,446.114505,274.209558,224.808528,412.062770,...,153.094814,231.418235,499.451812,505.914866,400.264984,443.208824,494.362277,522.567568,533.020189,344.070912
2013-11-01 00:10:00+01:00,119.137933,131.195257,271.487113,437.499259,556.889415,566.605046,414.028165,226.547075,183.504999,327.148452,...,161.125299,246.240862,426.869227,509.542921,411.589878,411.013982,485.583974,602.576329,666.040825,345.268477
2013-11-01 00:20:00+01:00,132.895668,145.327862,202.293674,395.703468,542.566906,554.649417,448.444528,276.578602,204.431304,302.434460,...,143.309675,197.684020,380.244161,502.357664,456.388379,381.148071,547.458650,594.929169,571.452502,301.364847
2013-11-01 00:30:00+01:00,111.203897,128.850880,180.342735,423.301776,607.967384,599.920977,453.085028,292.311112,212.374048,431.629557,...,123.002219,174.966267,382.336141,458.648782,371.216147,386.280515,434.587045,481.095703,507.696292,267.053464
2013-11-01 00:40:00+01:00,197.078066,309.570955,217.508334,357.359874,504.644640,549.331487,441.509067,236.566495,195.768005,310.414299,...,132.229509,178.626955,389.721354,552.115869,435.629009,387.856806,370.719776,549.667573,644.216408,332.631107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-11-21 23:10:00+01:00,160.606599,199.448377,242.501996,397.193527,497.654349,520.730540,478.073896,331.096738,223.577617,401.768500,...,262.005764,425.721698,596.048549,634.485506,825.260365,730.354938,697.511428,709.773705,756.076812,466.380082
2013-11-21 23:20:00+01:00,167.689205,223.785570,229.425744,425.494125,559.919007,613.492882,523.105600,353.423073,222.842564,374.644775,...,208.464229,368.892509,832.925759,670.544177,648.973157,712.780341,634.262872,740.047158,795.558607,563.590204
2013-11-21 23:30:00+01:00,150.242566,202.516317,269.874119,494.587265,632.359813,562.673415,398.265954,294.270733,241.201876,388.139502,...,224.512608,395.293158,827.779053,776.946103,802.778248,658.553458,614.262266,558.833184,597.291604,561.078394


In [104]:
tam_eksen = pd.date_range(pivot.index.min(),pivot.index.max(),freq="10min")

In [105]:
pivot_internet.reindex(tam_eksen)

square_id,4151,4152,4153,4154,4155,4156,4157,4158,4159,4160,...,6061,6062,6063,6064,6065,6066,6067,6068,6069,6070
2013-11-01 00:00:00+01:00,120.654932,138.451424,215.463804,435.474639,600.853735,598.512774,446.114505,274.209558,224.808528,412.062770,...,153.094814,231.418235,499.451812,505.914866,400.264984,443.208824,494.362277,522.567568,533.020189,344.070912
2013-11-01 00:10:00+01:00,119.137933,131.195257,271.487113,437.499259,556.889415,566.605046,414.028165,226.547075,183.504999,327.148452,...,161.125299,246.240862,426.869227,509.542921,411.589878,411.013982,485.583974,602.576329,666.040825,345.268477
2013-11-01 00:20:00+01:00,132.895668,145.327862,202.293674,395.703468,542.566906,554.649417,448.444528,276.578602,204.431304,302.434460,...,143.309675,197.684020,380.244161,502.357664,456.388379,381.148071,547.458650,594.929169,571.452502,301.364847
2013-11-01 00:30:00+01:00,111.203897,128.850880,180.342735,423.301776,607.967384,599.920977,453.085028,292.311112,212.374048,431.629557,...,123.002219,174.966267,382.336141,458.648782,371.216147,386.280515,434.587045,481.095703,507.696292,267.053464
2013-11-01 00:40:00+01:00,197.078066,309.570955,217.508334,357.359874,504.644640,549.331487,441.509067,236.566495,195.768005,310.414299,...,132.229509,178.626955,389.721354,552.115869,435.629009,387.856806,370.719776,549.667573,644.216408,332.631107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-11-21 23:10:00+01:00,160.606599,199.448377,242.501996,397.193527,497.654349,520.730540,478.073896,331.096738,223.577617,401.768500,...,262.005764,425.721698,596.048549,634.485506,825.260365,730.354938,697.511428,709.773705,756.076812,466.380082
2013-11-21 23:20:00+01:00,167.689205,223.785570,229.425744,425.494125,559.919007,613.492882,523.105600,353.423073,222.842564,374.644775,...,208.464229,368.892509,832.925759,670.544177,648.973157,712.780341,634.262872,740.047158,795.558607,563.590204
2013-11-21 23:30:00+01:00,150.242566,202.516317,269.874119,494.587265,632.359813,562.673415,398.265954,294.270733,241.201876,388.139502,...,224.512608,395.293158,827.779053,776.946103,802.778248,658.553458,614.262266,558.833184,597.291604,561.078394
2013-11-21 23:40:00+01:00,151.735856,194.750295,222.998701,454.597693,602.957549,544.025501,421.535300,306.571557,240.533536,430.124099,...,198.124052,298.136149,512.883000,600.482192,773.529617,585.920635,554.698894,546.205662,575.720316,424.449335


In [106]:
pivot_internet.isna().sum().sum()

np.int64(0)

In [107]:
pivot_internet.shape

(3024, 400)

In [108]:
pivot.index.to_series().diff().value_counts()

time_interval
0 days 00:10:00    3023
Name: count, dtype: int64

In [109]:
hourly = pivot_internet.resample('1h').sum()

In [110]:
hourly.shape

(504, 400)